# Exploratory Data Analysis: English → Darija (`imomayiz/darija-english`)
Run from the project root (next to `data.py`). Column names are auto-detected by `data.py`.

In [ ]:
!pip install -q datasets sentencepiece matplotlib pandas

In [ ]:
import re, collections
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from data import load_pairs, train_spm, load_raw_df, resolve_columns, HF_ID

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

## 1. Loading the raw dataset

In [ ]:
raw_df = load_raw_df()      # all splits concatenated
SRC_COL, TGT_COL = resolve_columns(raw_df)
print("Columns:", raw_df.columns.tolist(), "| using:", SRC_COL, "->", TGT_COL)
print("Rows:", len(raw_df))
raw_df.head()

## 2. Cleaning statistics

In [ ]:
df = raw_df[[SRC_COL, TGT_COL]].copy()
df.columns = ["src", "tgt"]

n_raw = len(df)
n_null = int(df.isna().any(axis=1).sum())
df = df.dropna()
df["src"] = df["src"].astype(str).str.strip()
df["tgt"] = df["tgt"].astype(str).str.strip()
n_empty = int(((df.src == "") | (df.tgt == "")).sum())
df = df[(df.src != "") & (df.tgt != "")]
n_dup_pairs = int(df.duplicated().sum())
n_dup_src = int(df.duplicated("src").sum())
n_dup_tgt = int(df.duplicated("tgt").sum())
df = df.drop_duplicates().reset_index(drop=True)

print(f"Raw rows               : {n_raw:,}")
print(f"Rows with nulls        : {n_null:,}")
print(f"Empty strings          : {n_empty:,}")
print(f"Duplicate (src,tgt)    : {n_dup_pairs:,}")
print(f"Same src, other tgt    : {n_dup_src:,}  (one English sentence, several Darija variants)")
print(f"Same tgt, other src    : {n_dup_tgt:,}")
print(f"Clean pairs            : {len(df):,}")

## 3. Script of the Darija side (Arabic vs Latin/“Arabizi”)
This decides the tokenizer behaviour and how noisy the spelling is.

In [ ]:
AR = re.compile(r"[\u0600-\u06FF]")
LAT = re.compile(r"[A-Za-z]")
# Arabizi digits used as letters: 2=ء 3=ع 5=خ 7=ح 9=ق  (e.g. "3lach", "7ta", "9alb")
ARABIZI_DIGIT = re.compile(r"(?<=[A-Za-z])[2-9]|[2-9](?=[A-Za-z])")

def arabic_ratio(s):
    a, l = len(AR.findall(s)), len(LAT.findall(s))
    return a / (a + l) if (a + l) else np.nan

df["tgt_ar_ratio"] = df.tgt.map(arabic_ratio)
df["src_ar_ratio"] = df.src.map(arabic_ratio)
df["tgt_arabizi_digits"] = df.tgt.map(lambda s: bool(ARABIZI_DIGIT.search(s)))

print("Target mostly Arabic script :", f"{(df.tgt_ar_ratio > 0.5).mean():.1%}")
print("Target mostly Latin script  :", f"{(df.tgt_ar_ratio <= 0.5).mean():.1%}")
print("Target mixed (0.1-0.9)      :", f"{df.tgt_ar_ratio.between(0.1, 0.9).mean():.1%}")
print("Target with Arabizi digits  :", f"{df.tgt_arabizi_digits.mean():.1%}")
print("Source with Arabic chars    :", f"{(df.src_ar_ratio > 0).mean():.1%}  (should be ~0)")

df.tgt_ar_ratio.hist(bins=20)
plt.xlabel("Share of Arabic-script letters in Darija sentence"); plt.ylabel("# sentences")
plt.title("Script distribution of the target"); plt.show()

## 4. Sentence length (words)

In [ ]:
df["src_w"] = df.src.str.split().map(len)
df["tgt_w"] = df.tgt.str.split().map(len)
df["src_c"] = df.src.str.len()
df["tgt_c"] = df.tgt.str.len()

pcts = [50, 75, 90, 95, 99]
summary = pd.DataFrame({
    "English words": df.src_w.describe(),
    "Darija words": df.tgt_w.describe(),
}).T
summary[[f"p{p}" for p in pcts]] = [[np.percentile(d, p) for p in pcts] for d in (df.src_w, df.tgt_w)]
display(summary.round(2))

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6), sharey=True)
clip = int(max(np.percentile(df.src_w, 99), np.percentile(df.tgt_w, 99)))
ax[0].hist(df.src_w.clip(upper=clip), bins=40, color="#4c72b0"); ax[0].set_title("English (words)")
ax[1].hist(df.tgt_w.clip(upper=clip), bins=40, color="#dd8452"); ax[1].set_title("Darija (words)")
for a in ax: a.set_xlabel("words per sentence (clipped at p99)")
ax[0].set_ylabel("# sentences"); plt.tight_layout(); plt.show()

## 5. Subword tokens and the `max_len` choice
SentencePiece BPE is trained on the **training split only** (same function and split as the training pipeline).

In [ ]:
VOCAB, MAX_LEN = 8000, 64
train_pairs, val_pairs, test_pairs = load_pairs(seed=42)
print(f"Split sizes -> train {len(train_pairs):,} | val {len(val_pairs):,} | test {len(test_pairs):,}")

import os, sentencepiece as spm
os.makedirs("spm", exist_ok=True)
sp_src = train_spm([s for s, _ in train_pairs], "spm/src", VOCAB)
sp_tgt = train_spm([t for _, t in train_pairs], "spm/tgt", VOCAB)

all_pairs = train_pairs + val_pairs + test_pairs
src_len = np.array([len(sp_src.encode(s)) + 1 for s, _ in all_pairs])      # + EOS
tgt_len = np.array([len(sp_tgt.encode(t)) + 2 for _, t in all_pairs])      # + BOS/EOS

print(f"Source tokens: mean {src_len.mean():.1f}, p95 {np.percentile(src_len,95):.0f}, p99 {np.percentile(src_len,99):.0f}, max {src_len.max()}")
print(f"Target tokens: mean {tgt_len.mean():.1f}, p95 {np.percentile(tgt_len,95):.0f}, p99 {np.percentile(tgt_len,99):.0f}, max {tgt_len.max()}")
for L in (32, 48, 64, 96):
    cov = ((src_len <= L) & (tgt_len <= L)).mean()
    print(f"max_len={L:3d} keeps {cov:.1%} of pairs untruncated")

plt.hist([src_len, tgt_len], bins=40, label=["English", "Darija"], color=["#4c72b0", "#dd8452"])
plt.axvline(MAX_LEN, color="k", ls="--", label=f"max_len={MAX_LEN}")
plt.xlabel("subword tokens"); plt.ylabel("# sentences"); plt.legend(); plt.title("Token lengths"); plt.show()

print("Fertility (tokens per word): EN %.2f | Darija %.2f" % (
    src_len.sum() / df.src_w.sum(), tgt_len.sum() / df.tgt_w.sum()))

## 6. Vocabulary

In [ ]:
WORD = re.compile(r"\w+", re.UNICODE)
def words(texts): return [w.lower() for t in texts for w in WORD.findall(t)]

src_tr = collections.Counter(words(s for s, _ in train_pairs))
tgt_tr = collections.Counter(words(t for _, t in train_pairs))

def vocab_stats(c, name):
    total = sum(c.values()); V = len(c)
    hapax = sum(1 for v in c.values() if v == 1)
    top100 = sum(v for _, v in c.most_common(100)) / total
    print(f"{name:7s} tokens {total:>9,} | types {V:>7,} | hapax {hapax/V:.1%} of types | top-100 words cover {top100:.1%}")

vocab_stats(src_tr, "English"); vocab_stats(tgt_tr, "Darija")

display(pd.DataFrame({
    "English": [f"{w} ({n})" for w, n in src_tr.most_common(15)],
    "Darija":  [f"{w} ({n})" for w, n in tgt_tr.most_common(15)],
}))

for c, lab in ((src_tr, "English"), (tgt_tr, "Darija")):
    f = np.array(sorted(c.values(), reverse=True))
    plt.loglog(np.arange(1, len(f) + 1), f, label=lab)
plt.xlabel("rank"); plt.ylabel("frequency"); plt.title("Zipf plot (train)"); plt.legend(); plt.show()

## 7. Spelling variability in Darija
Darija has no standard orthography. As a rough proxy, collapse repeated letters (`mchiiii` → `mchi`) and see how much the vocabulary shrinks; a big drop means many noisy variants.

In [ ]:
collapse = lambda w: re.sub(r"(.)\1+", r"\1", w)
for c, lab in ((src_tr, "English"), (tgt_tr, "Darija")):
    collapsed = {collapse(w) for w in c}
    print(f"{lab:8s} types {len(c):,} -> {len(collapsed):,} after collapsing repeats "
          f"({1 - len(collapsed)/len(c):.1%} reduction)")

# Example of one Darija 'word' with many spellings (grouped by repeat-collapsed form)
groups = collections.defaultdict(list)
for w, n in tgt_tr.items(): groups[collapse(w)].append((w, n))
multi = sorted((g for g in groups.values() if len(g) > 2), key=lambda g: -sum(n for _, n in g))[:8]
for g in multi: print(sorted(g, key=lambda x: -x[1])[:6])

## 8. Split sanity: OOV rate and leakage

In [ ]:
def oov_rate(counter, texts):
    toks = words(texts)
    return np.mean([w not in counter for w in toks])

for name, pairs in (("val", val_pairs), ("test", test_pairs)):
    print(f"{name:4s} OOV rate  EN {oov_rate(src_tr, [s for s,_ in pairs]):.2%} | "
          f"Darija {oov_rate(tgt_tr, [t for _,t in pairs]):.2%}")

train_src = {s.lower() for s, _ in train_pairs}
train_tgt = {t.lower() for _, t in train_pairs}
for name, pairs in (("val", val_pairs), ("test", test_pairs)):
    ls = np.mean([s.lower() in train_src for s, _ in pairs])
    lt = np.mean([t.lower() in train_tgt for _, t in pairs])
    print(f"{name:4s} source seen in train: {ls:.2%} | target seen in train: {lt:.2%}")
print("\nIf those leakage numbers are noticeable, consider grouping the split by source sentence.")

## 9. Alignment sanity check (length ratio)

In [ ]:
df["ratio"] = (df.tgt_w + 1) / (df.src_w + 1)
print(df.ratio.describe().round(2))

plt.hist(df.ratio.clip(upper=4), bins=60); plt.axvline(1, color="k", ls="--")
plt.xlabel("Darija words / English words (clipped at 4)"); plt.ylabel("# pairs")
plt.title("Length ratio"); plt.show()

sus = df[(df.ratio > 3) | (df.ratio < 1/3)]
print(f"Suspicious pairs (ratio >3 or <1/3): {len(sus):,} ({len(sus)/len(df):.2%})")
display(sus[["src", "tgt", "ratio"]].head(8))

## 10. Random samples

In [ ]:
pd.set_option("display.max_colwidth", 120)
df[["src", "tgt"]].sample(12, random_state=0)

## 11. Summary (fill in your conclusions)

In [ ]:
print("=== Key numbers for the report ===")
print(f"Clean pairs: {len(df):,} | train/val/test: {len(train_pairs):,}/{len(val_pairs):,}/{len(test_pairs):,}")
print(f"Median words EN/Darija: {df.src_w.median():.0f}/{df.tgt_w.median():.0f}")
print(f"Darija mostly Arabic script: {(df.tgt_ar_ratio > 0.5).mean():.1%}")
print(f"Pairs fully covered by max_len={MAX_LEN}: {((src_len<=MAX_LEN)&(tgt_len<=MAX_LEN)).mean():.1%}")
print(f"Darija vocab {len(tgt_tr):,} types vs English {len(src_tr):,}")

**Conclusions to write in the report (edit after running):**
- Dataset size, split, and cleaning applied.
- Script of the Darija side and what it implies (noisy spelling → BPE subwords, chrF alongside BLEU).
- Length statistics → why `max_len=64` (or your value).
- Vocabulary size / OOV → why SentencePiece with 8k units.
- Leakage and alignment risks you found.